# smFISH: MS2 infection prevalence and classification threshold

Experiment 1: Figures 2D, S10. Experiments 1 and 2: Table S3.

Environment: `python_env` (`envs/python_env.yml`)

Inputs: `data/smFISH_exp1_per_cell.csv`, `data/smFISH_exp2_per_cell.csv`, one row per segmented cell, from `preprocessing/smFISH_image_processing.py`

Within each experiment, cells are MS2-positive if their whole-cell signal density exceeds the maximum of that experiment's negative-control cells.

In [ ]:
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT = Path("..")
config = yaml.safe_load(open(ROOT / "config.yaml"))
OUT_DIR = Path(config["out_dir"])
FIG_DIR = OUT_DIR / "figures"
SRC_DIR = OUT_DIR / "source_data"
FIG_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

INK = "#1a1a1a"
DOT_GRAY = "#c2c2c2"
RULE_GRAY = "#d9d9d9"

mpl.rcParams.update({
    "pdf.fonttype": 42,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 9,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 1.0,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "xtick.major.size": 3.0,
    "ytick.major.size": 3.0,
    "ytick.minor.width": 0.8,
    "ytick.minor.size": 1.4,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "text.color": INK,
    "axes.labelcolor": INK,
    "axes.edgecolor": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "figure.dpi": 150,
})


def save_fig(fig, name, dpi=600):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=dpi, bbox_inches="tight", pad_inches=0.02, facecolor="white")
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight", pad_inches=0.02, facecolor="white")


def wilson_ci(k, n, z=1.959963984540054):
    if n <= 0:
        return np.nan, np.nan
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return max(0.0, centre - half), min(1.0, centre + half)

In [ ]:
GROUP_LABELS = {
    "Negative control": "NC",
    "Experimental t0": "0",
    "t15": "15",
    "t45": "45",
    "t120": "120",
    "t180": "180",
    "t240": "240",
}
GROUP_ORDER = list(GROUP_LABELS)
CONTROL_GROUP = "Negative control"
METRIC = "whole_cell_signal_density"


def load_cells(path):
    df = pd.read_csv(path)
    assert not df.duplicated(["image_name", "cell_id"]).any()
    assert df[METRIC].notna().all()
    df["analysis_group"] = np.where(df.sample_type.eq("negative_control"), CONTROL_GROUP,
                                    np.where(df.timepoint.eq("t0"), "Experimental t0", df.timepoint))
    assert set(df.analysis_group) == set(GROUP_ORDER), set(df.analysis_group) ^ set(GROUP_ORDER)
    df["analysis_group"] = pd.Categorical(df.analysis_group, categories=GROUP_ORDER, ordered=True)

    negative = df.loc[df.analysis_group == CONTROL_GROUP, METRIC]
    threshold = float(np.nextafter(negative.max(), np.inf))
    df["ms2_positive"] = df[METRIC] >= threshold

    pooled = (df.groupby("analysis_group", observed=True)
                .agg(n_images=("image_name", "nunique"), n_cells=("cell_id", "count"),
                     n_infected=("ms2_positive", "sum"))
                .reset_index())
    pooled["prevalence"] = 100 * pooled.n_infected / pooled.n_cells
    ci = [wilson_ci(k, n) for k, n in zip(pooled.n_infected, pooled.n_cells)]
    pooled["ci_low"] = [100 * c[0] for c in ci]
    pooled["ci_high"] = [100 * c[1] for c in ci]

    per_image = (df.groupby(["analysis_group", "image_name"], observed=True)
                   .agg(n_cells=("cell_id", "count"), n_infected=("ms2_positive", "sum"))
                   .reset_index())
    per_image["prevalence"] = 100 * per_image.n_infected / per_image.n_cells
    return df, threshold, pooled, per_image


data = {exp: load_cells(ROOT / "data" / f"smFISH_{exp}_per_cell.csv") for exp in ("exp1", "exp2")}
for exp, (d, thr, _, _) in data.items():
    print(f"{exp}: {d.image_name.nunique()} fields, {len(d)} cells, threshold = {thr:.4f}")
df, threshold, pooled, per_image = data["exp1"]

In [ ]:
X_GAPS = {"Experimental t0": 0, "t15": 8, "t45": 14, "t120": 18, "t180": 18, "t240": 18}
CONTROL_GAP = 8
xpos = dict(zip(X_GAPS, np.cumsum(list(X_GAPS.values())).astype(float)))
xpos[CONTROL_GROUP] = xpos["Experimental t0"] - CONTROL_GAP
jitter_sd = 0.06 * np.diff(np.sort(list(xpos.values()))).min()


def plot_prevalence(pooled, per_image, name):
    rng = np.random.default_rng(2026)
    fig, ax = plt.subplots(figsize=(3.6, 2.3))
    rng = np.random.default_rng(2026)

    for group in GROUP_ORDER:
        vals = per_image.loc[per_image.analysis_group == group, "prevalence"].to_numpy(dtype=float)
        if vals.size:
            ax.scatter(xpos[group] + rng.normal(0, jitter_sd, size=vals.size), vals,
                       s=14, color=DOT_GRAY, alpha=0.7, linewidths=0, zorder=2)

    rows = pooled.set_index("analysis_group")
    course = [g for g in GROUP_ORDER if g != CONTROL_GROUP]
    ax.plot([xpos[g] for g in course], [rows.loc[g, "prevalence"] for g in course],
            color=INK, linewidth=1.8, solid_capstyle="round", zorder=3)

    for group in GROUP_ORDER:
        r = rows.loc[group]
        p = float(r.prevalence)
        is_control = group == CONTROL_GROUP
        ax.errorbar(xpos[group], p, yerr=[[max(p - r.ci_low, 0.0)], [max(r.ci_high - p, 0.0)]],
                    fmt="o", markersize=5.5, color=INK,
                    markerfacecolor="white" if is_control else INK,
                    markeredgecolor=INK if is_control else "white",
                    markeredgewidth=1.2 if is_control else 0.6,
                    elinewidth=1.3, capsize=2.5, capthick=1.3, zorder=4)

    tick_x = [xpos[g] for g in GROUP_ORDER]
    ax.set_xticks(tick_x)
    ax.set_xticklabels([GROUP_LABELS[g] for g in GROUP_ORDER])
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.set_ylim(-4, 104)
    ax.set_ylabel("Infected cells (%)")
    ax.set_xlabel("Time (minutes post infection)")
    x_pad = 0.035 * (max(tick_x) - min(tick_x))
    ax.set_xlim(min(tick_x) - x_pad, max(tick_x) + x_pad)
    ax.axvline((xpos[CONTROL_GROUP] + xpos["Experimental t0"]) / 2,
               color=RULE_GRAY, linewidth=0.9, linestyle=(0, (2, 2)), zorder=1)
    for side in ("top", "right", "bottom", "left"):
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(0.5)
    ax.tick_params(top=False, right=False)

    save_fig(fig, name)
    plt.show()
    pooled.to_csv(SRC_DIR / f"{name}_pooled.csv", index=False)
    per_image.to_csv(SRC_DIR / f"{name}_per_image.csv", index=False)

In [ ]:
def plot_threshold(df, threshold, name):
    negative = df.loc[df.analysis_group == CONTROL_GROUP, METRIC]
    values = {
        "Negative control": negative.to_numpy(),
        "0 min": df.loc[df.analysis_group == "Experimental t0", METRIC].to_numpy(),
        "15 min": df.loc[df.analysis_group == "t15", METRIC].to_numpy(),
    }
    colors = {"Negative control": "#1F77B4", "0 min": "#7F7F7F", "15 min": "#FF7F0E"}
    q995 = np.quantile(negative, 0.995)

    pooled_vals = np.concatenate(list(values.values()))
    lo = min(0.0, pooled_vals.min())
    hi = min(pooled_vals.max(), max(threshold * 6.0, q995 * 1.25))
    bins = np.linspace(lo, hi, 71)

    with plt.rc_context({"font.size": 16, "axes.labelsize": 18, "xtick.labelsize": 15, "ytick.labelsize": 15,
                         "legend.fontsize": 14, "axes.linewidth": 1.2, "xtick.major.size": 5, "ytick.major.size": 5}):
        fig, ax = plt.subplots(figsize=(8, 5.5))
        for g, v in values.items():
            ax.hist(v, bins=bins, density=True, histtype="step", linewidth=2.2, color=colors[g])
        ax.axvline(threshold, color="black", linewidth=2.2)
        ax.set_xlim(lo, hi)
        ax.set_xlabel("Whole-cell MS2 signal density (a.u.)")
        ax.set_ylabel("Probability density")
        handles = [Line2D([0], [0], color=colors[g], lw=2.2, label=f"{g} (n = {len(v):,})") for g, v in values.items()]
        handles.append(Line2D([0], [0], color="black", lw=2.2, label=f"Threshold = {threshold:.3f}"))
        ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)
        save_fig(fig, name, dpi=300)
        plt.show()

    pd.DataFrame({"group": np.repeat(list(values), [len(v) for v in values.values()]),
                  METRIC: pooled_vals}).to_csv(SRC_DIR / f"{name}_data.csv", index=False)

## Figure 2D

In [ ]:
plot_prevalence(pooled, per_image, "Fig2D")

## Figure S10

In [ ]:
plot_threshold(df, threshold, "FigS10")

## Table S3

In [ ]:
TAB_DIR = OUT_DIR / "tables"
TAB_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for i, (exp, (d, thr, pl, _)) in enumerate(data.items(), start=1):
    t = pl.assign(experiment=f"Experiment {i}", time_point=pl.analysis_group.map(GROUP_LABELS))
    rows.append(t[["experiment", "time_point", "n_images", "n_cells", "n_infected", "prevalence"]])
    d.to_csv(SRC_DIR / f"smFISH_{exp}_per_cell_classified.csv", index=False)
s3 = pd.concat(rows, ignore_index=True)
s3.columns = ["Experiment", "Time point (min)", "Fields", "Cells", "MS2-positive cells", "MS2-positive (%)"]
s3 = s3.round({"MS2-positive (%)": 1})
s3.to_csv(TAB_DIR / "TableS3.csv", index=False)
s3